In [4]:




import os
import glob
import pandas as pd

# 源数据目录
SRC_DIR = r"C:\Users\greenlandProject\ERA5LAND\dataverse_files\month"

# 目标年份
TARGET_YEAR = 2025

# 输出到桌面
OUTPUT_DIR = r"C:\Users\11823\Desktop"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"albedo_{TARGET_YEAR}_all_stations.csv")


def extract_station_name(filename: str) -> str:
    base = os.path.basename(filename)
    name, _ = os.path.splitext(base)
    lower = name.lower()
    key = "_month"
    pos = lower.find(key)
    return name[:pos] if pos != -1 else name


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_rows = []

    csv_files = glob.glob(os.path.join(SRC_DIR, "*.csv"))
    if not csv_files:
        print(f"目录内未发现csv文件：{SRC_DIR}")
        return

    for fp in csv_files:
        station = extract_station_name(fp)

        try:
            df = pd.read_csv(fp)
        except Exception as e:
            print(f"文件读取失败：{fp}，原因：{e}")
            continue

        if "time" not in df.columns:
            print(f"文件缺少time列，已跳过：{fp}")
            continue
        if "albedo" not in df.columns:
            print(f"文件缺少albedo列，已跳过：{fp}")
            continue

        # 解析时间，去掉已弃用的 infer_datetime_format 参数
        dt = pd.to_datetime(df["time"], errors="coerce")
        mask = dt.dt.year == TARGET_YEAR

        # 过滤目标年份
        df_year = df.loc[mask, ["albedo"]].copy()
        df_year["year"] = TARGET_YEAR
        df_year["month"] = dt.loc[mask].dt.month
        df_year.insert(0, "station", station)

        # 仅保留四列且顺序固定
        df_year = df_year[["station", "year", "month", "albedo"]]

        count_vals = len(df_year)
        all_rows.append(df_year)

        print(f"站点{station}{TARGET_YEAR}年有{count_vals}个值，已保存")

    if not all_rows:
        print(f"{TARGET_YEAR}年无任何站点数据，未生成汇总文件。")
        return

    result = pd.concat(all_rows, ignore_index=True)
    result.sort_values(by=["station", "year", "month"], inplace=True)

    result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(f"汇总文件已生成：{OUTPUT_FILE}（共{len(result)}行）")


if __name__ == "__main__":
    main()

站点CEN2025年有9个值，已保存
站点CP12025年有9个值，已保存
站点DY22025年有9个值，已保存
站点EGP2025年有9个值，已保存
站点FRE2025年有9个值，已保存
站点HUM2025年有9个值，已保存
站点JAR2025年有9个值，已保存
站点KAN_B2025年有9个值，已保存
站点KAN_L2025年有9个值，已保存
站点KAN_M2025年有9个值，已保存
站点KAN_T2025年有9个值，已保存
站点KAN_U2025年有9个值，已保存
站点KPC_L2025年有9个值，已保存
站点KPC_U2025年有9个值，已保存
站点LYN_L2025年有9个值，已保存
站点LYN_T2025年有0个值，已保存
站点MIT2025年有9个值，已保存
站点NAE2025年有9个值，已保存
站点NAU2025年有9个值，已保存
站点NEM2025年有9个值，已保存
站点NSE2025年有9个值，已保存
站点NUK_B2025年有8个值，已保存
站点NUK_K2025年有9个值，已保存
站点NUK_L2025年有9个值，已保存
站点NUK_N2025年有0个值，已保存
站点NUK_U2025年有9个值，已保存
站点QAS_A2025年有0个值，已保存
站点QAS_L2025年有9个值，已保存
站点QAS_M2025年有4个值，已保存
站点QAS_U2025年有9个值，已保存
站点RED_L2025年有9个值，已保存
站点SCO_L2025年有9个值，已保存
站点SCO_U2025年有9个值，已保存
站点SDL2025年有9个值，已保存
站点SDM2025年有9个值，已保存
站点SER_B2025年有9个值，已保存
站点SWC2025年有9个值，已保存
站点TAS_A2025年有9个值，已保存
站点TAS_L2025年有9个值，已保存
站点TAS_U2025年有0个值，已保存
站点THU_L22025年有7个值，已保存
站点THU_L2025年有8个值，已保存
站点THU_U2025年有9个值，已保存
站点TUN2025年有9个值，已保存
站点UPE_L2025年有9个值，已保存
站点UPE_U2025年有9个值，已保存
站点WEG_B2025年有9个值，已保存
站点WEG_L2025年有9个值，已保存
站点ZAC_A2025年有7个值，已保存
站点

In [6]:
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

def merge_nc_files(era5_path, snow_path, output_path):
    print("正在读取数据文件...")
    ds_era5 = xr.open_dataset(era5_path, decode_times=False, chunks="auto")
    ds_snow = xr.open_dataset(snow_path, decode_times=False, chunks="auto")

    # 坐标名统一
    if "longitude" in ds_era5.coords: ds_era5 = ds_era5.rename({"longitude":"lon"})
    if "longitude" in ds_snow.coords: ds_snow = ds_snow.rename({"longitude":"lon"})
    if "latitude"  in ds_era5.coords: ds_era5 = ds_era5.rename({"latitude":"time"})
    if "latitude"  in ds_snow.coords: ds_snow = ds_snow.rename({"latitude":"time"})

    # snow 经度 360 → [-180,180)
    snow_lon = ds_snow.lon
    ds_snow = ds_snow.assign_coords(lon=((snow_lon + 180) % 360) - 180).sortby("lon")

    # 确保纬度(time)升序
    if ds_era5.time[0] > ds_era5.time[-1]:
        ds_era5 = ds_era5.sortby("time")
    if ds_snow.time[0] > ds_snow.time[-1]:
        ds_snow = ds_snow.sortby("time")

    # 插值对齐到 ERA5 的 lon/time/valid_time
    interp_targets = {"lon": ds_era5.lon, "time": ds_era5.time}
    if "valid_time" in ds_era5.coords and "valid_time" in ds_snow.coords:
        interp_targets["valid_time"] = ds_era5.valid_time
    ds_snow = ds_snow.interp(interp_targets, method="nearest")

    # 强制赋值坐标（逐值完全一致）
    assign_coords = {"lon": ds_era5.lon, "time": ds_era5.time}
    if "valid_time" in ds_era5.coords and "valid_time" in ds_snow.coords:
        assign_coords["valid_time"] = ds_era5.valid_time
    ds_snow = ds_snow.assign_coords(assign_coords)

    # 检查一致性
    assert np.array_equal(ds_snow.lon.values, ds_era5.lon.values), "lon 未对齐"
    assert np.array_equal(ds_snow.time.values, ds_era5.time.values), "time 未对齐"
    if "valid_time" in assign_coords:
        assert np.array_equal(ds_snow.valid_time.values, ds_era5.valid_time.values), "valid_time 未对齐"

    # 删除可能冲突的额外坐标
    for extra in ["expver","number"]:
        if extra in ds_era5.coords and extra not in ds_era5.dims:
            ds_era5 = ds_era5.drop_vars(extra)
        if extra in ds_snow.coords and extra not in ds_snow.dims:
            ds_snow = ds_snow.drop_vars(extra)

    # 同名变量冲突：保留 ERA5
    conflicts = set(ds_era5.data_vars) & set(ds_snow.data_vars)
    if conflicts:
        print(f"⚠ 冲突变量保留 ERA5: {sorted(conflicts)}")
        ds_snow = ds_snow.drop_vars(sorted(conflicts))

    # 合并
    ds_merged = xr.merge([ds_era5, ds_snow], compat="no_conflicts", combine_attrs="override")
    assert ds_merged.dims["lon"] == ds_era5.dims["lon"], "lon 维度不一致"
    assert ds_merged.dims["time"] == ds_era5.dims["time"], "time 维度不一致"

    # 保存
    enc = {v: {"zlib": True, "complevel": 4} for v in ds_merged.data_vars}
    ds_merged.to_netcdf(output_path, engine="netcdf4", encoding=enc)

    ds_era5.close(); ds_snow.close(); ds_merged.close()
    return output_path

def verify_merged_file(merged_path):
    print(f"\n验证合并文件: {merged_path}")
    ds = xr.open_dataset(merged_path)
    print("维度:", dict(ds.dims))
    print("变量数:", len(ds.data_vars))
    print("坐标:", list(ds.coords))

    missing = {}
    for v in ds.data_vars:
        n = int(np.asarray(ds[v].isnull().sum().values).item())
        if n > 0:
            missing[v] = n
    print("NaN 统计:", missing if missing else "无缺失")
    ds.close()

def check_subregion(merged_path, center_lat=72.5, center_lon=-40, size=100):
    """
    从中心点抽取 size×size 的网格检查是否有 NaN
    """
    ds = xr.open_dataset(merged_path)

    # 子区域范围
    half = size//2
    lat_vals = ds.time.values
    lon_vals = ds.lon.values

    lat_center_idx = np.argmin(np.abs(lat_vals - center_lat))
    lon_center_idx = np.argmin(np.abs(lon_vals - center_lon))

    lat_slice = slice(lat_vals[max(0, lat_center_idx-half)],
                      lat_vals[min(len(lat_vals)-1, lat_center_idx+half)])
    lon_slice = slice(lon_vals[max(0, lon_center_idx-half)],
                      lon_vals[min(len(lon_vals)-1, lon_center_idx+half)])

    sub = ds.isel(valid_time=0).sel(time=lat_slice, lon=lon_slice)

    print(f"\n抽查子区域: lat≈{center_lat}, lon≈{center_lon}, 尺寸≈{size}×{size}")
    for v in sub.data_vars:
        n = int(np.asarray(sub[v].isnull().sum().values).item())
        print(f"  {v}: NaN={n} / {sub[v].size}")

    ds.close()

if __name__ == "__main__":
    era5_file = Path(r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\ERA5LANDfeature2025.nc")
    snow_file = Path(r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\snow3para2025.nc")
    output_file = Path(r"C:\Users\11823\Downloads\merged_era5_snow_2025.nc")

    if not era5_file.exists() or not snow_file.exists():
        raise SystemExit("❌ 输入文件不存在")

    result = merge_nc_files(str(era5_file), str(snow_file), str(output_file))
    verify_merged_file(result)
    check_subregion(result, center_lat=72.5, center_lon=-40, size=100)
    print("\n🎉 合并与检查完成！")


正在读取数据文件...


C:\Users\11823\AppData\Local\Temp\ipykernel_17588\1034617237.py:60: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert ds_merged.dims["lon"] == ds_era5.dims["lon"], "lon 维度不一致"
C:\Users\11823\AppData\Local\Temp\ipykernel_17588\1034617237.py:61: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert ds_merged.dims["time"] == ds_era5.dims["time"], "time 维度不一致"



验证合并文件: C:\Users\11823\Downloads\merged_era5_snow_2025.nc
维度: {'valid_time': 8, 'time': 351, 'lon': 751}
变量数: 12
坐标: ['valid_time', 'time', 'lon']


C:\Users\11823\AppData\Local\Temp\ipykernel_17588\1034617237.py:73: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print("维度:", dict(ds.dims))


NaN 统计: {'t2m': 1357808, 'asn': 1357808, 'fal': 1357808, 'slhf': 1357808, 'str': 1357808, 'ssrd': 1357808, 'strd': 1357808, 'u10': 1357808, 'v10': 1357808, 'sp': 1357808, 'snowc': 1357808, 'sde': 1357808}

抽查子区域: lat≈72.5, lon≈-40, 尺寸≈100×100
  t2m: NaN=0 / 10201
  asn: NaN=0 / 10201
  fal: NaN=0 / 10201
  slhf: NaN=0 / 10201
  str: NaN=0 / 10201
  ssrd: NaN=0 / 10201
  strd: NaN=0 / 10201
  u10: NaN=0 / 10201
  v10: NaN=0 / 10201
  sp: NaN=0 / 10201
  snowc: NaN=0 / 10201
  sde: NaN=0 / 10201

🎉 合并与检查完成！


In [11]:
import pandas as pd
import xarray as xr

# 文件路径
file_csv = r"C:\Users\11823\Desktop\albedo_2025_all_stations.csv"
file_nc = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\merged_era5_snow_2025.nc"

# 1. 读取 CSV 和 NetCDF
df = pd.read_csv(file_csv)
ds = xr.open_dataset(file_nc)

# 2. 重命名 time -> lat（因为它其实是纬度）
ds = ds.rename({"time": "lat"})

# 3. 构造 valid_time（每月1号即可）
df['valid_time'] = pd.to_datetime(df[['year', 'month']].assign(day=1))

# 4. 获取所有变量名（12个）
variables = list(ds.data_vars)
print("✅ 将要提取的变量：", variables)

# 5. 遍历 CSV，提取最近邻气象变量
results = []
for idx, row in df.iterrows():
    point_data = ds.sel(
        valid_time=row['valid_time'],
        lat=row['latitude'],
        lon=row['longitude'],
        method="nearest"
    )
    values = {var: point_data[var].values.item() for var in variables}
    results.append(values)

# 6. 把结果合并到 df
df_vars = pd.DataFrame(results)
df_final = pd.concat([df, df_vars], axis=1)

# 7. 保存回原文件（覆盖保存）
df_final.to_csv(file_csv, index=False)

print("🎉 已完成：文件已更新，新增 12 列气象变量。")


✅ 将要提取的变量： ['t2m', 'asn', 'fal', 'slhf', 'str', 'ssrd', 'strd', 'u10', 'v10', 'sp', 'snowc', 'sde']
🎉 已完成：文件已更新，新增 12 列气象变量。


In [12]:
import pandas as pd
import numpy as np

def merge_altitude_data():
    """
    根据station列匹配，将merged_allfeature.csv中的altitude_last_valid列
    添加到albedo_2025_all_stations.csv中
    """
    
    try:
        # 读取两个CSV文件
        print("正在读取文件...")
        
        # 读取包含altitude_last_valid的文件
        df_altitude = pd.read_csv(r"C:\Users\greenlandProject\ERA5LAND\merged_allfeature.csv")
        print(f"merged_allfeature.csv: {df_altitude.shape[0]} 行, {df_altitude.shape[1]} 列")
        
        # 读取需要添加altitude列的文件
        df_albedo = pd.read_csv(r"C:\Users\11823\Desktop\albedo_2025_all_stations.csv")
        print(f"albedo_2025_all_stations.csv: {df_albedo.shape[0]} 行, {df_albedo.shape[1]} 列")
        
        # 检查必要的列是否存在
        if 'station' not in df_altitude.columns:
            print("错误: merged_allfeature.csv 中没有找到 'station' 列")
            return
        
        if 'station' not in df_albedo.columns:
            print("错误: albedo_2025_all_stations.csv 中没有找到 'station' 列")
            return
            
        if 'altitude_last_valid' not in df_altitude.columns:
            print("错误: merged_allfeature.csv 中没有找到 'altitude_last_valid' 列")
            return
        
        # 显示station列的基本信息
        print(f"\nmerged_allfeature.csv 中唯一station数量: {df_altitude['station'].nunique()}")
        print(f"albedo_2025_all_stations.csv 中唯一station数量: {df_albedo['station'].nunique()}")
        
        # 创建altitude数据的映射表（去重，每个station只保留一个altitude值）
        altitude_mapping = df_altitude[['station', 'altitude_last_valid']].drop_duplicates(subset=['station'])
        
        # 检查是否有station对应多个不同的altitude值
        station_altitude_counts = df_altitude.groupby('station')['altitude_last_valid'].nunique()
        multi_altitude_stations = station_altitude_counts[station_altitude_counts > 1]
        
        if len(multi_altitude_stations) > 0:
            print(f"\n警告: {len(multi_altitude_stations)} 个station有多个不同的altitude_last_valid值:")
            for station, count in multi_altitude_stations.head().items():
                altitudes = df_altitude[df_altitude['station'] == station]['altitude_last_valid'].unique()
                print(f"  Station {station}: {count} 个不同值 {altitudes}")
            print("  将使用每个station的第一个altitude值")
        
        # 执行左连接，根据station列匹配
        df_merged = df_albedo.merge(
            altitude_mapping, 
            on='station', 
            how='left'
        )
        
        # 检查匹配结果
        matched_stations = df_merged['altitude_last_valid'].notna().sum()
        total_stations = len(df_merged)
        
        print(f"\n匹配结果:")
        print(f"总记录数: {total_stations}")
        print(f"成功匹配altitude的记录数: {matched_stations}")
        print(f"未匹配到altitude的记录数: {total_stations - matched_stations}")
        print(f"匹配率: {matched_stations/total_stations*100:.1f}%")
        
        # 显示未匹配的station（如果数量不多的话）
        unmatched_stations = df_merged[df_merged['altitude_last_valid'].isna()]['station'].unique()
        if len(unmatched_stations) > 0 and len(unmatched_stations) <= 10:
            print(f"\n未匹配到altitude的station: {list(unmatched_stations)}")
        elif len(unmatched_stations) > 10:
            print(f"\n未匹配到altitude的station太多({len(unmatched_stations)}个)，不完全显示")
        
        # 保存合并后的文件
        output_path = r"C:\Users\11823\Desktop\albedo_2025_all_stations_with_altitude.csv"
        df_merged.to_csv(output_path, index=False)
        print(f"\n合并后的文件已保存到: {output_path}")
        print(f"最终文件: {df_merged.shape[0]} 行, {df_merged.shape[1]} 列")
        
        # 显示altitude列的统计信息
        if matched_stations > 0:
            altitude_stats = df_merged['altitude_last_valid'].describe()
            print(f"\naltitude_last_valid列统计信息:")
            print(altitude_stats)
        
        return df_merged
        
    except FileNotFoundError as e:
        print(f"文件未找到错误: {e}")
        print("请检查文件路径是否正确")
    except Exception as e:
        print(f"处理过程中出现错误: {e}")

# 执行函数
if __name__ == "__main__":
    result_df = merge_altitude_data()

正在读取文件...
merged_allfeature.csv: 4455 行, 17 列
albedo_2025_all_stations.csv: 412 行, 18 列

merged_allfeature.csv 中唯一station数量: 49
albedo_2025_all_stations.csv 中唯一station数量: 47

匹配结果:
总记录数: 412
成功匹配altitude的记录数: 395
未匹配到altitude的记录数: 17
匹配率: 95.9%

未匹配到altitude的station: ['NUK_B', 'SER_B']

合并后的文件已保存到: C:\Users\11823\Desktop\albedo_2025_all_stations_with_altitude.csv
最终文件: 412 行, 19 列

altitude_last_valid列统计信息:
count     395.000000
mean     1159.748785
std       799.079962
min        17.900000
25%       551.230000
50%       876.710000
75%      1920.390000
max      2897.530000
Name: altitude_last_valid, dtype: float64


In [14]:
import pandas as pd
import os

def process_albedo_file(file_path):
    """
    Reads a CSV file, rounds the longitude and latitude to one decimal place,
    and saves the modified data to a new file with "_0.1" appended to the name.

    Args:
        file_path (str): The full path to the input CSV file.
    """
    try:
        # 1. 检查文件路径是否存在
        if not os.path.exists(file_path):
            print(f"Error: The file at {file_path} does not exist.")
            return

        # 2. 读取CSV文件
        df = pd.read_csv(file_path)

        # 3. 检查文件是否包含 'longitude' 和 'latitude' 列
        if 'longitude' not in df.columns or 'latitude' not in df.columns:
            print("Error: The CSV file must contain 'longitude' and 'latitude' columns.")
            return

        # 4. 对经纬度列进行四舍五入，保留一位小数
        # 我们可以使用 .round() 方法
        df['longitude'] = df['longitude'].round(1)
        df['latitude'] = df['latitude'].round(1)

        # 5. 构建新文件的路径
        dir_name = os.path.dirname(file_path)
        base_name = os.path.basename(file_path)
        file_name, file_ext = os.path.splitext(base_name)
        new_file_path = os.path.join(dir_name, f"{file_name}_0.1{file_ext}")

        # 6. 保存修改后的数据到新文件
        df.to_csv(new_file_path, index=False)
        
        print(f"File processed successfully! The new file is saved at: {new_file_path}")

    except Exception as e:
        print(f"An error occurred: {e}")

# 调用函数，传入你的文件路径
# 请将下面的路径替换成你的实际文件路径
file_path = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations.csv"
process_albedo_file(file_path)

File processed successfully! The new file is saved at: C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1.csv


In [20]:
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import pearsonr
import os

# ========== 1. 路径 ==========
file_path = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1.csv"
model_dir = r"C:\Users\11823\era5land\saved_models_917"  # 你保存pkl文件的目录

# 需要的特征
input_features = [
    "longitude", "latitude", "altitude_last_valid",
    "sp", "ssrd", "str", "strd",
    "t2m", "u10", "v10",
    "snowc", "sde", "asn",
    "year", "month"
]

# ========== 2. 加载数据 ==========
df = pd.read_csv(file_path)

X_new = df[input_features].astype(float)
y_true = df["st_albedo"].astype(float)  # 真值列

# ========== 3. 加载模型 ==========
base_models = {
    'RF': joblib.load(os.path.join(model_dir, "RF_917.pkl")),
    'XGB': joblib.load(os.path.join(model_dir, "XGB_917.pkl")),
    'GBDT': joblib.load(os.path.join(model_dir, "GBDT_917.pkl")),
    'LGBM': joblib.load(os.path.join(model_dir, "LGBM_917.pkl")),
    'CatBoost': joblib.load(os.path.join(model_dir, "CatBoost_917.pkl")),
}
meta_model = joblib.load(os.path.join(model_dir, "meta_model_917.pkl"))

# ========== 4. 预测 ==========
# 先用每个基础模型预测
meta_features_new = np.column_stack([m.predict(X_new) for m in base_models.values()])

# 再用元模型融合
y_pred = meta_model.predict(meta_features_new)

# 存入DataFrame
df["albedo_pred"] = y_pred

# ========== 5. 计算统计量 ==========
r2 = r2_score(y_true, y_pred)
r, _ = pearsonr(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
rmse = np.sqrt(mse)
bias = np.mean(y_pred - y_true)

print("\n========== 2025 Prediction Evaluation ==========")
print(f"R²: {r2:.6f}")
print(f"Pearson R: {r:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"Bias: {bias:.6f}")

# ========== 6. 保存结果 ==========
out_path = file_path.replace(".csv", "_with_pred_917.csv")
df.to_csv(out_path, index=False)
print(f"\n✅ 预测结果已保存到: {out_path}")



========== 2025 Prediction Evaluation ==========
R²: 0.808971
Pearson R: 0.902202
RMSE: 0.092454
MSE: 0.008548
Bias: 0.007030

✅ 预测结果已保存到: C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1_with_pred_917.csv


In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import pearsonr, spearmanr
import os

# ===== 1) 配置：替换为你的结果文件路径（需包含 st_albedo 与 albedo_pred 两列）=====
file_path = r"C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1_with_pred_917.csv"

# ===== 2) 读取与清洗 =====
df = pd.read_csv(file_path)
if not {"st_albedo", "albedo_pred"}.issubset(df.columns):
    raise ValueError("CSV 必须包含列：'st_albedo' 和 'albedo_pred'。")

# 丢掉无效值
df = df.copy()
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["st_albedo", "albedo_pred"])
y_true = df["st_albedo"].astype(float).values
y_pred = df["albedo_pred"].astype(float).values

# ===== 3) 计算指标 =====
def lin_ccc(x, y):
    """Lin's Concordance Correlation Coefficient"""
    mu_x, mu_y = np.mean(x), np.mean(y)
    var_x, var_y = np.var(x, ddof=1), np.var(y, ddof=1)
    sd_x, sd_y = np.sqrt(var_x), np.sqrt(var_y)
    r, _ = pearsonr(x, y)
    return (2 * r * sd_x * sd_y) / (var_x + var_y + (mu_x - mu_y) ** 2)

r2 = r2_score(y_true, y_pred)
r, _ = pearsonr(y_true, y_pred)
rho, _ = spearmanr(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
bias = np.mean(y_pred - y_true)                          # MBE
nmbe_pct = bias / np.mean(y_true) * 100                  # 归一化偏差（相对均值）
nrmse_pct = rmse / np.mean(y_true) * 100                 # 归一化RMSE（相对均值）
ccc = lin_ccc(y_true, y_pred)

# 简单的OLS拟合（y_pred ~ a * y_true + b）
a, b = np.polyfit(y_true, y_pred, 1)

metrics = {
    "N": [len(y_true)],
    "R2": [r2],
    "Pearson_R": [r],
    "Spearman_rho": [rho],
    "RMSE": [rmse],
    "MAE": [mae],
    "MBE_Bias": [bias],
    "NMBE_%": [nmbe_pct],
    "NRMSE_%": [nrmse_pct],
    "Lin_CCC": [ccc],
    "OLS_slope": [a],
    "OLS_intercept": [b],
    "y_true_mean": [np.mean(y_true)],
    "y_pred_mean": [np.mean(y_pred)],
}

metrics_df = pd.DataFrame(metrics)
print("\n========== Metrics (2025) ==========")
print(metrics_df.to_string(index=False))

# ===== 4) 保存指标CSV =====
out_dir = os.path.dirname(file_path)
metrics_path = os.path.join(out_dir, os.path.basename(file_path).replace(".csv", "_metrics_917.csv"))
metrics_df.to_csv(metrics_path, index=False)
print(f"\n✅ 指标已保存: {metrics_path}")

# ===== 5) 散点图（带 1:1 线 + 回归线）=====
plt.figure(figsize=(7, 7))
plt.scatter(y_true, y_pred, s=12, alpha=0.6)

# 1:1 参考线
xy_min = float(np.nanmin([y_true.min(), y_pred.min()]))
xy_max = float(np.nanmax([y_true.max(), y_pred.max()]))
plt.plot([xy_min, xy_max], [xy_min, xy_max], linestyle="--")

# 回归线
x_line = np.linspace(xy_min, xy_max, 100)
y_line = a * x_line + b
plt.plot(x_line, y_line, linestyle="-")

plt.xlabel("st_albedo (Truth)")
plt.ylabel("albedo_pred (Model)")
plt.title("st_albedo vs albedo_pred (with 1:1 and OLS fit)")

# 在图上标注关键指标
txt = (
    f"N={len(y_true)}\n"
    f"R²={r2:.3f}\n"
    f"R={r:.3f}, ρ={rho:.3f}\n"
    f"RMSE={rmse:.3f}, MAE={mae:.3f}\n"
    f"Bias={bias:.3f}\n"
    f"NMBE={nmbe_pct:.2f}%, NRMSE={nrmse_pct:.2f}%\n"
    f"CCC={ccc:.3f}\n"
    f"slope={a:.3f}, intercept={b:.3f}"
)
plt.gca().text(0.05, 0.95, txt, transform=plt.gca().transAxes, va="top", ha="left",
               bbox=dict(boxstyle="round", alpha=0.2))

plt.tight_layout()
fig_path = os.path.join(out_dir, os.path.basename(file_path).replace(".csv", "_scatter_917.png"))
plt.savefig(fig_path, dpi=200)
plt.close()
print(f"✅ 散点图已保存: {fig_path}")



========== Metrics (2025) ==========
  N       R2  Pearson_R  Spearman_rho     RMSE      MAE  MBE_Bias   NMBE_%   NRMSE_%  Lin_CCC  OLS_slope  OLS_intercept  y_true_mean  y_pred_mean
228 0.808971   0.902202      0.868102 0.092454 0.060193   0.00703 1.042496 13.709432 0.888054   0.757683       0.170445     0.674384     0.681414

✅ 指标已保存: C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1_with_pred_917_metrics_917.csv
✅ 散点图已保存: C:\Users\greenlandProject\ERA5LAND\ProcessFile\albedo_2025_all_stations_0.1_with_pred_917_scatter_917.png
